In [1]:
import torch
import wandb
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune
import time

from training import train_epoch, eval_epoch

from models import TeacherCNN, StudentCNN, BigCNN

In [2]:
from mnist1d_dataset import get_mnist1d_loaders, get_dataset_info

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader, test_loader = get_mnist1d_loaders()

info = get_dataset_info()
print(info)

File already exists. Skipping download.
Successfully loaded data from ./mnist1d_data.pkl
File already exists. Skipping download.
Successfully loaded data from ./mnist1d_data.pkl
{'n_train': 4000, 'n_test': 1000, 'input_length': 40, 'num_classes': 10}


## Knoledge Distillation

### Teacher model

In [ ]:
wandb.init(
    project="mnist-1d-KDvsLTH",
    group="TeacherCNN",
    name="TeacherCNN",
    config={
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "epochs": 500,
        "patience": 50,
        "architecture": "TeacherCNN",
        "scheduler": "CosineAnnealingLR"
    }
)

teacher = TeacherCNN().to(device)
optimizer = torch.optim.Adam(teacher.parameters(), lr=wandb.config.learning_rate, weight_decay=wandb.config.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=wandb.config.epochs)
criterion = nn.CrossEntropyLoss()
epochs = wandb.config.epochs
#early_stopper = EarlyStopping(patience=wandb.config.patience)

wandb.watch(teacher, log_freq=100)

best_test_acc = 0.0
model_path = "models/best_teacher_model.pth"

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(teacher, train_loader, optimizer, criterion, device)
    test_loss, test_acc = eval_epoch(teacher, test_loader, criterion, device)

    # Step the scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    wandb.log({
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "learning_rate": current_lr
    }, step=epoch)

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(teacher.state_dict(), model_path)
        wandb.save(model_path)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:03d} | Acc: {test_acc:.3f} | LR: {current_lr:.6f}")

print("\nTraining complete.")

teacher.load_state_dict(torch.load(model_path))

final_loss, final_acc = eval_epoch(teacher, test_loader, nn.CrossEntropyLoss(), device)

wandb.log({
    "best_test_loss": final_loss,
    "best_test_acc": final_acc
})

print(f"Best Teacher model - Acc: {final_acc:.4f} | Loss: {final_loss:.4f}")

wandb.finish()

In [5]:
model_path = "models/best_teacher_model.pth"
teacher = TeacherCNN().to(device)
teacher.load_state_dict(torch.load(model_path))

teacher_loss, teacher_acc = eval_epoch(teacher, test_loader, nn.CrossEntropyLoss(),device=device)

print(f"Best Teacher model - Acc: {teacher_acc:.4f} | Loss: {teacher_loss:.4f}")

Best Teacher model - Acc: 0.9950 | Loss: 0.0265


### Student model

#### Student Baseline: No Distillation

In [7]:
wandb.init(
    project="mnist-1d-KDvsLTH",
    name="StudentCNN_Baseline(no_KD)",
    group="StudentCNN",
    config={
        "learning_rate": 1e-2,
        "weight_decay": 1e-4,
        "epochs": 500,
        "patience": 50,
        "architecture": "StudentCNN",
        "scheduler": "CosineAnnealingLR"
    }
)

student = StudentCNN().to(device)
optimizer = torch.optim.Adam(student.parameters(), lr=wandb.config.learning_rate, weight_decay=wandb.config.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=wandb.config.epochs)
criterion = nn.CrossEntropyLoss()
epochs = wandb.config.epochs
#early_stopper = EarlyStopping(patience=wandb.config.patience)

best_test_acc = 0.0
try:
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(student, train_loader, optimizer, criterion, device)
        test_loss, test_acc = eval_epoch(student, test_loader, criterion, device)
        scheduler.step()

        wandb.log({
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_loss,
            "test_acc": test_acc,
        }, step=epoch)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Train acc: {train_acc:.3f} | "
            f"Test acc: {test_acc:.3f}"
            )
        
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            torch.save(student.state_dict(), "models/best_baseline_student_model.pth")
            wandb.save("models/best_baseline_student_model.pth")
finally:
    student.load_state_dict(torch.load("models/best_baseline_student_model.pth"))
    final_loss, final_acc = eval_epoch(student, test_loader, nn.CrossEntropyLoss(), device)
    wandb.log({
        "best_test_loss": final_loss,
        "best_test_acc": final_acc
    })
    print(f"Best Student model - Acc: {final_acc:.4f} | Loss: {final_loss:.4f}")
    wandb.finish()


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 01 | Train acc: 0.336 | Test acc: 0.541
Epoch 02 | Train acc: 0.621 | Test acc: 0.637
Epoch 03 | Train acc: 0.712 | Test acc: 0.719
Epoch 04 | Train acc: 0.767 | Test acc: 0.729
Epoch 05 | Train acc: 0.803 | Test acc: 0.781
Epoch 06 | Train acc: 0.842 | Test acc: 0.835
Epoch 07 | Train acc: 0.874 | Test acc: 0.832
Epoch 08 | Train acc: 0.864 | Test acc: 0.813
Epoch 09 | Train acc: 0.903 | Test acc: 0.883
Epoch 10 | Train acc: 0.921 | Test acc: 0.868
Epoch 11 | Train acc: 0.938 | Test acc: 0.868
Epoch 12 | Train acc: 0.938 | Test acc: 0.866
Epoch 13 | Train acc: 0.959 | Test acc: 0.867
Epoch 14 | Train acc: 0.936 | Test acc: 0.882
Epoch 15 | Train acc: 0.949 | Test acc: 0.885
Epoch 16 | Train acc: 0.967 | Test acc: 0.904
Epoch 17 | Train acc: 0.969 | Test acc: 0.904
Epoch 18 | Train acc: 0.968 | Test acc: 0.896
Epoch 19 | Train acc: 0.970 | Test acc: 0.879
Epoch 20 | Train acc: 0.972 | Test acc: 0.888
Epoch 21 | Train acc: 0.974 | Test acc: 0.903
Epoch 22 | Train acc: 0.975 | Test

best_test_acc,▁
best_test_loss,▁
test_acc,▁▁▄▃▆▆▅▆▆▇▇▇▇▇███████████████▇██████████
test_loss,▆█▄▇▄▂▂▃▂▂▅▂▃▂▂▆▁▁▂▃▃▂▂▂▁▂▁▁▁▁▂▂▂▁▁▁▁▁▁▁
train_acc,▁████▇██████████████████████████████████
train_loss,█▂▁▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_test_acc,0.962
best_test_loss,0.14699
test_acc,0.958
test_loss,0.15326
train_acc,1


In [8]:
student = StudentCNN().to(device)
student.load_state_dict(torch.load("models/best_baseline_student_model.pth"))
final_loss, final_acc = eval_epoch(student, test_loader, nn.CrossEntropyLoss(), device=device)
print(f"Best Student model - Acc: {final_acc:.4f} | Loss: {final_loss:.4f}")

Best Student model - Acc: 0.9620 | Loss: 0.1470


#### Knoledge Distilled Student

In [3]:
from training import train_kd_epoch, eval_kd_epoch

model_path = "models/best_teacher_model.pth"
teacher = TeacherCNN().to(device)
teacher.load_state_dict(torch.load(model_path))

wandb.init(
    project="mnist-1d-KDvsLTH",
    name="KD-Student",
    group="StudentCNN",
    config={
        "learning_rate": 1e-2,
        "weight_decay": 1e-4,
        "T_start": 12,
        "T_end": 8,
        "alpha": 0.2,
        "epochs": 500,
        "patience": 50,
        "architecture": "StudentCNN",
        "scheduler": "CosineAnnealingLR"
    }
)

student = StudentCNN().to(device)
optimizer = torch.optim.Adam(student.parameters(), lr=wandb.config.learning_rate, weight_decay=wandb.config.weight_decay)
epochs = wandb.config.epochs
T_start = wandb.config.T_start  # 10 from your config
T_end = wandb.config.T_end
alpha = wandb.config.alpha  # weighting between hard and soft loss
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)


best_test_acc = 0.0
start_time = time.time()
try:
    for epoch in range(epochs):
        # temperature decay
        T = T_start - (T_start - T_end) * (epoch / (epochs - 1))
        # Train student with KD
        train_loss, train_acc = train_kd_epoch(student, teacher, train_loader, optimizer, T=T, alpha=alpha, device=device)

        # Evaluate student with KD (teacher provided)
        test_loss, test_acc = eval_kd_epoch(student, teacher, test_loader, T=T, alpha=alpha, device=device)

        scheduler.step()

        wandb.log({
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_loss,
            "test_acc": test_acc,
            "temperature": T
        }, step=epoch)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Train acc: {train_acc:.3f} | "
            f"Test acc: {test_acc:.3f}"
        )
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            torch.save(student.state_dict(), "models/best_kd_student_model.pth")
            wandb.save("models/best_kd_student_model.pth")

finally:
    training_time = time.time() - start_time
    wandb.log({
        "total_training_time_sec": training_time,
        "total_training_time_min": training_time / 60
    })
    print(f"\n⏱ Total training time: {training_time/60:.2f} minutes")
    student.load_state_dict(torch.load("models/best_kd_student_model.pth"))
    final_loss, final_acc = eval_epoch(student, test_loader, nn.CrossEntropyLoss(), device=device)
    wandb.log({
        "best_test_loss": final_loss,
        "best_test_acc": final_acc
    })
    print(f"Best Knoledge Distilled Student model - Acc: {final_acc:.4f} | Loss: {final_loss:.4f}")
    wandb.finish()


wandb: Currently logged in as: matteo-piras (matteo-piras-universit-di-firenze) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 01 | Train acc: 0.236 | Test acc: 0.315
Epoch 02 | Train acc: 0.445 | Test acc: 0.543
Epoch 03 | Train acc: 0.635 | Test acc: 0.683
Epoch 04 | Train acc: 0.713 | Test acc: 0.718
Epoch 05 | Train acc: 0.759 | Test acc: 0.787
Epoch 06 | Train acc: 0.806 | Test acc: 0.814
Epoch 07 | Train acc: 0.844 | Test acc: 0.852
Epoch 08 | Train acc: 0.869 | Test acc: 0.870
Epoch 09 | Train acc: 0.890 | Test acc: 0.866
Epoch 10 | Train acc: 0.915 | Test acc: 0.877
Epoch 11 | Train acc: 0.922 | Test acc: 0.899
Epoch 12 | Train acc: 0.931 | Test acc: 0.892
Epoch 13 | Train acc: 0.936 | Test acc: 0.903
Epoch 14 | Train acc: 0.943 | Test acc: 0.917
Epoch 15 | Train acc: 0.950 | Test acc: 0.887
Epoch 16 | Train acc: 0.955 | Test acc: 0.920
Epoch 17 | Train acc: 0.960 | Test acc: 0.921
Epoch 18 | Train acc: 0.964 | Test acc: 0.926
Epoch 19 | Train acc: 0.963 | Test acc: 0.921
Epoch 20 | Train acc: 0.970 | Test acc: 0.932
Epoch 21 | Train acc: 0.969 | Test acc: 0.933
Epoch 22 | Train acc: 0.971 | Test

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 500 | Train acc: 0.999 | Test acc: 0.962

⏱ Total training time: 2.12 minutes
Best Knoledge Distilled Student model - Acc: 0.9700 | Loss: 0.1519


best_test_acc,▁
best_test_loss,▁
temperature,█████▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁
test_acc,▁▂▆▇████████████████████████████████████
test_loss,██▆▄▃▄▃▂▂▃▂▃▂▂▂▂▂▃▂▂▁▂▂▂▁▂▂▁▁▁▂▁▂▁▁▁▁▁▁▁
total_training_time_min,▁
total_training_time_sec,▁
train_acc,▁▇▇▇▇███████████████████████████████████
train_loss,█▅▅▅▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_test_acc,0.97
best_test_loss,0.1519


In [4]:
student = StudentCNN().to(device)
student.load_state_dict(torch.load("models/best_kd_student_model.pth"))
kd_loss, kd_acc = eval_epoch(student, test_loader, nn.CrossEntropyLoss(), device=device)
print(f"Best Knoledge Distilled Student model - Acc: {kd_acc:.4f} | Loss: {kd_loss:.4f}")

Best Knoledge Distilled Student model - Acc: 0.9700 | Loss: 0.1519


## Lottery Ticket Hypotesis

In [7]:
from lth_utils import *
import torch.nn.utils.prune as prune
import copy

def run_lth(train_loader, test_loader, rounds=5, prune_amount=0.2, epochs_per_round=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    lth_start_time = time.time()
    
    # Initialize Network
    model = BigCNN().to(device)
    wandb.watch(model, log="all")
    
    # Save Initial State (Theta_0)
    initial_state = copy.deepcopy(model.state_dict())

    cumulative_time = 0.0
    
    print(f"Initial parameters: {sum(p.numel() for p in model.parameters())}")

    for round_idx in range(rounds):
        print(f"\n--- Round {round_idx + 1} ---")

        round_start_time = time.time()
        
        # Train (Re-init optimizer to reset momentum)
        optimizer = optim.Adam(model.parameters(), lr=0.001) 
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs_per_round)
        
        history = train_model(
        model, 
        device, 
        train_loader, 
        test_loader, 
        optimizer,
        scheduler,
        epochs=epochs_per_round,
        round_idx=round_idx  # <--- Add this line
        )

        round_time = time.time() - round_start_time
        cumulative_time += round_time

        best_acc = max(history['val_acc'])

        # Calculate Sparsity
        sparsity_pct = count_sparsity(model)
        print(f"Sparsity after pruning: {sparsity_pct:.2f}%")
        active_params = count_active_parameters(model)
        print(f"Active parameters after pruning: {active_params}")

        # Prune
        parameters_to_prune = get_prunable_layers(model)
        prune.global_unstructured(
            parameters_to_prune,
            pruning_method=prune.L1Unstructured,
            amount=prune_amount,
        )

        wandb.log({
            "active_parameters": active_params,
            "sparsity_percentage": sparsity_pct,
            "round": round_idx,
            "best_val_acc": best_acc,
            "round_time_min": round_time/60,
            "cumulative_lth_time_min": cumulative_time/60
            })
        
        # Reset Weights (The Rewind)
        with torch.no_grad():
            for name, param in model.named_parameters():
                clean_name = name.replace("_orig", "")
                if clean_name in initial_state:
                     param.data.copy_(initial_state[clean_name].data)

        print("Weights reset to initialization.")

    remove_pruning_reparam(model)

    total_lth_time = time.time() - lth_start_time

    wandb.log({
        "total_training_time_sec": total_lth_time,
        "total_training_time_min": total_lth_time / 60
    })

    print(f"\n⏱ Total Training time: {total_lth_time/60:.2f} minutes")

    print(f"LTH Pruned model - Acc: {final_acc:.4f} | Loss: {final_loss:.4f}")
    return model, final_acc, final_loss, total_lth_time

In [17]:
# Initialize your project
wandb.init(
    project="mnist-1d-KDvsLTH",
    name="LTH-CNN",
    group="LTH-CNN",
    config={
        "learning_rate": 0.001,
        "rounds": 10,
        "prune_amount": 0.2,
        "epochs_per_round": 200,
        "architecture": "BigCNN"
    }
)


# Assuming you have your loaders ready
pruned_model, lth_acc, lth_loss, total_lth_time = run_lth(
    train_loader, 
    test_loader, 
    rounds=wandb.config.rounds, 
    prune_amount=wandb.config.prune_amount, 
    epochs_per_round=wandb.config.epochs_per_round
)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


active_parameters,█▆▅▄▃▂▂▁▁
best_val_acc,█▅▂▅▆▅▄▅▁
cumulative_lth_time_min,▁▂▃▄▄▅▆▇█
round,▁▂▃▄▅▅▆▇█
round_0/train_loss,█▇▅▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
round_0/val_acc,▁▅▅▅▇▇██████████████████████████████████
round_1/train_loss,█▇▆▄▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
round_1/val_acc,▁▅▅▆▇███████████████████████████████████
round_2/train_loss,█▇▆▄▄▃▃▃▃▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
round_2/val_acc,▁▅▅▇▇▇███▇██████████████████████████████
+16,...


Initial parameters: 60938

--- Round 1 ---
Round 0 | Epoch 1 | Val Acc: 37.90%
Round 0 | Epoch 2 | Val Acc: 63.60%
Round 0 | Epoch 3 | Val Acc: 69.80%
Round 0 | Epoch 4 | Val Acc: 75.30%
Round 0 | Epoch 5 | Val Acc: 73.70%
Round 0 | Epoch 6 | Val Acc: 74.30%
Round 0 | Epoch 7 | Val Acc: 75.10%
Round 0 | Epoch 8 | Val Acc: 75.10%
Round 0 | Epoch 9 | Val Acc: 76.00%
Round 0 | Epoch 10 | Val Acc: 79.60%
Round 0 | Epoch 11 | Val Acc: 83.20%
Round 0 | Epoch 12 | Val Acc: 81.20%
Round 0 | Epoch 13 | Val Acc: 84.20%
Round 0 | Epoch 14 | Val Acc: 86.20%
Round 0 | Epoch 15 | Val Acc: 92.00%
Round 0 | Epoch 16 | Val Acc: 91.10%
Round 0 | Epoch 17 | Val Acc: 92.00%
Round 0 | Epoch 18 | Val Acc: 92.40%
Round 0 | Epoch 19 | Val Acc: 92.30%
Round 0 | Epoch 20 | Val Acc: 92.40%
Round 0 | Epoch 21 | Val Acc: 92.70%
Round 0 | Epoch 22 | Val Acc: 92.00%
Round 0 | Epoch 23 | Val Acc: 92.40%
Round 0 | Epoch 24 | Val Acc: 93.40%
Round 0 | Epoch 25 | Val Acc: 93.20%
Round 0 | Epoch 26 | Val Acc: 93.70%
Roun

In [34]:
from models import count_parameters
dense_model = BigCNN().to(device)

dense_model.load_state_dict(torch.load("best_lth_model_round_0.pth", map_location=device))
dense_model_loss, dense_model_acc = eval_epoch(dense_model, test_loader, nn.CrossEntropyLoss(), device=device)
print(f"Dense model - Acc: {dense_model_acc:.2f} | Loss: {dense_model_loss:.4f} | Parameters: {count_parameters(dense_model)}")

Dense model - Acc: 0.99 | Loss: 0.0329 | Parameters: 60938


In [32]:
model = BigCNN().to(device)

parameters_to_prune = get_prunable_layers(model)
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.2,
)

model.load_state_dict(torch.load("best_lth_model_round_9.pth", map_location=device))
lth_loss, lth_acc = eval_epoch(model, test_loader, nn.CrossEntropyLoss(), device=device)
print(f"LTH Pruned model - Acc: {lth_acc:.4f} | Loss: {lth_loss:.4f}")
# Calculate Sparsity
sparsity_pct = count_sparsity(model)
print(f"Sparsity after pruning: {sparsity_pct:.2f}%")
active_params = count_active_parameters(model)
print(f"Active parameters after pruning: {active_params}")

LTH Pruned model - Acc: 0.9840 | Loss: 0.0617
Sparsity after pruning: 86.58%
Layer: features.0           | Active:      121 /      160 (75.62%)
Layer: features.5           | Active:      885 /    10240 (8.64%)
Layer: features.10          | Active:     1992 /    40960 (4.86%)
Layer: classifier.1         | Active:     4557 /     8192 (55.63%)
Layer: classifier.4         | Active:      523 /      640 (81.72%)
------------------------------------------------------------
TOTAL ACTIVE PARAMETERS: 8,078 / 60,192
GLOBAL DENSITY: 13.42%
Active parameters after pruning: 8078


In [33]:
from models import count_parameters
from lth_utils import count_sparsity, count_active_parameters
print("KNOLEDGE DISTILLATION VS LOTTERY TICKET HYPOTHESIS TRAINING TIME COMPARISON")
print("Knoledge distillation:")
print(f"Teacher - accuracy: {teacher_acc:.2f} | teacher parameters: {count_parameters(teacher)}")
print(f"Student - accuracy: {kd_acc:.2f} | training time: {training_time/60:.2f} minutes | student parameters: {count_parameters(student)}")
print("Lottery Ticket Hypothesis:")
print(f"Dense model - Acc: {dense_model_acc:.2f} | parameters: {count_parameters(dense_model)}")
print(f"Lottery Ticket- accuracy: {lth_acc:.2f} | training time: {total_lth_time/60:.2f} minutes | active parameters: {active_params} | sparsity: {sparsity_pct:.2f}%")

KNOLEDGE DISTILLATION VS LOTTERY TICKET HYPOTHESIS TRAINING TIME COMPARISON
Knoledge distillation:
Teacher - accuracy: 0.99 | teacher parameters: 999562
Student - accuracy: 0.97 | training time: 2.00 minutes | student parameters: 8059
Lottery Ticket Hypothesis:
Dense model - Acc: 0.99 | parameters: 60938
Lottery Ticket- accuracy: 0.98 | training time: 12.31 minutes | active parameters: 8078 | sparsity: 86.58%
